# Lab 2. Pixels, histograms, colour

**INFO-H-500 | 3 hours | Course link: image representation (sampling, quantization, colour), point processing**

You read an image through its histogram, change it with look-up tables (LUT), and select colours in HSV.

**Goals**
- Compute and read a histogram, a normalized histogram and a cumulative histogram.
- Write simple LUTs and see their effect on the histogram.
- Understand why HSV is easier than RGB for colour selection.
- Understand lossless compression with a pyramid of remainders.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color

def check(name, ok):
    print(('OK     ' if ok else 'CHECK  ') + name)

camera = data.camera()
camera_u8 = camera

## Exercise 1. Histograms and entropy

The histogram counts how many pixels have each gray level. For a `uint8` image there are 256 levels.

In [ ]:
# histogram of the camera image with np.bincount (256 bins, use minlength=256)
# Your code here
# normalized histogram p (sums to 1) and cumulative histogram cdf (ends at 1)
# Your code here

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(256), p, width=1.0)
axes[0].set_title('normalized histogram')
axes[1].plot(cdf)
axes[1].set_title('cumulative histogram')
plt.show()

check('histogram sums to the number of pixels', hist.sum() == camera.size)
check('p sums to 1', np.isclose(p.sum(), 1))
check('cdf ends at 1', np.isclose(cdf[-1], 1))

The entropy measures the information carried by the histogram, in bits per pixel:

$$H = -\sum_i p_i \log_2 p_i \qquad (p_i > 0)$$

In [ ]:
def entropy(p):
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

H = entropy(p)
print('entropy: %.3f bits' % H)

# shuffle the pixels of the image (np.random.default_rng(0).permutation on the flat array, then reshape)
# Your code here

plt.figure(figsize=(4, 4))
plt.imshow(shuffled, cmap='gray')
plt.show()
p2 = np.bincount(shuffled.ravel(), minlength=256) / camera.size
check('same histogram after shuffling', np.allclose(p, p2))
check('same entropy after shuffling', np.isclose(H, entropy(p2)))

**Answer 1.** The image is destroyed but the histogram and the entropy do not change. What does this tell you about what a histogram contains and what it does not?

*(your answer here)*

## Exercise 2. Look-up tables

A LUT is an array of 256 values. Applying it to an image is one line: `lut[image]`.

In [ ]:
levels = np.arange(256)

# negative: 255 - level
# Your code here
# threshold: 255 above 128, 0 otherwise (uint8)
# Your code here
# contrast stretch: map the 2nd percentile to 0 and the 98th to 255, clip outside (uint8)
# Your code here
# equalization: use the cumulative histogram cdf of the image as LUT (scaled to 0..255, uint8)
# Your code here

results = {'original': camera,
           'negative': lut_negative[camera],
           'threshold': lut_threshold[camera],
           'stretch': lut_stretch[camera],
           'equalized': lut_equal[camera]}

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, (name, im) in enumerate(results.items()):
    axes[0, i].imshow(im, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(name)
    axes[0, i].axis('off')
    axes[1, i].hist(im.ravel(), bins=64, range=(0, 255))
plt.tight_layout()
plt.show()

check('negative of 0 is 255', lut_negative[0] == 255)
check('stretch reaches 0 and 255', lut_stretch[0] == 0 and lut_stretch[255] == 255)
check('equalized histogram is flatter: cdf is close to a straight line',
      np.abs(np.cumsum(np.bincount(lut_equal[camera].ravel(), minlength=256)) / camera.size - np.linspace(0, 1, 256)).max() < 0.1)

**Answer 2.** Compare the five histograms. Which LUT loses information (two different levels become the same)? Which one can be inverted?

*(your answer here)*

## Exercise 3. Colour

The IHC image shows a colon tissue stained with a brown marker (DAB) and a blue counterstain.

In [ ]:
ihc = data.immunohistochemistry()
print(ihc.shape, ihc.dtype)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(['red', 'green', 'blue']):
    axes[i].hist(ihc[:, :, i].ravel(), bins=64, range=(0, 255), color=name)
    axes[i].set_title(name + ' histogram')
plt.show()
plt.figure(figsize=(5, 5))
plt.imshow(ihc)
plt.show()

In [ ]:
hsv = color.rgb2hsv(ihc)
hue, sat, val = hsv[..., 0], hsv[..., 1], hsv[..., 2]

# mask of the brown pixels in HSV: hue between 0.02 and 0.14, saturation above 0.35 (adjust by looking at the result)
# Your code here
# same goal in RGB: distance to a brown reference colour, mask = distance below a threshold
# Your code here

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(ihc)
axes[0].set_title('IHC')
axes[1].imshow(mask_hsv, cmap='gray')
axes[1].set_title('HSV mask: %.1f %% of the pixels' % (100 * mask_hsv.mean()))
axes[2].imshow(mask_rgb, cmap='gray')
axes[2].set_title('RGB distance mask: %.1f %%' % (100 * mask_rgb.mean()))
for ax in axes:
    ax.axis('off')
plt.show()

check('some pixels are selected, but not all', 0.01 < mask_hsv.mean() < 0.6)

**Answer 3.** Write two lines on why the HSV selection works better than the RGB one. Hint: change the lighting by multiplying the image by 0.6 and see what happens to each mask.

*(your answer here)*

## Exercise 4. Image compression: a pyramid of remainders

Neighbouring pixels have similar values. We use this to reduce the entropy **without losing any information**.

One level of the decomposition cuts the image into 2 x 2 groups of pixels. The four pixels of a group go to four images of half the size: $a$ (top-left pixel), $b$, $c$ and $d$. We keep $a$, and we replace $b$, $c$ and $d$ by their **remainders** with $a$: $b-a$, $c-a$ and $d-a$. These remainders are small, because the pixels are close. The next level does the same on $a$ only, and so on.

The remainders can be negative, so we work with signed integers (`int16`) to avoid overflow.

In [ ]:
def split(im):
    """One level: the four blocks a, b, c, d in a 2x2 mosaic, with b, c, d replaced by remainders."""
    a = im[0::2, 0::2]
    b = im[0::2, 1::2]
    c = im[1::2, 0::2]
    d = im[1::2, 1::2]
    # replace b, c and d by the remainders b - a, c - a, d - a
    # Your code here
    return np.vstack((np.hstack((a, b)), np.hstack((c, d))))

def pyramid(im, levels):
    """Apply split() on the top-left block, level after level."""
    out = im.astype(np.int16)           # signed: the remainders are negative
    size = out.shape[0]
    for _ in range(levels):
        out[:size, :size] = split(out[:size, :size])
        size //= 2
    return out

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, lev in zip(axes, [0, 1, 3]):
    ax.imshow(pyramid(camera_u8, lev), cmap='gray', vmin=-255, vmax=255, interpolation='nearest')
    ax.set_title('%d level(s)' % lev)
    ax.axis('off')
plt.show()

Compute the entropy of the whole decomposition (all the values of the array, counted together) for 0 to 6 levels.

In [ ]:
def entropy_of_values(x):
    _, counts = np.unique(x, return_counts=True)
    p = counts / x.size
    return -np.sum(p * np.log2(p))

# entropy of pyramid(camera_u8, levels) for levels = 0..6, in a list
# Your code here

plt.figure(figsize=(6, 3))
plt.plot(range(7), entropies, 'o-')
plt.xlabel('levels')
plt.ylabel('entropy (bits per value)')
plt.show()
print(np.round(entropies, 2))
check('the entropy of the original image is the one of exercise 1', np.isclose(entropies[0], H))
check('the entropy decreases with the first levels', entropies[1] < entropies[0] and entropies[2] < entropies[1])

**Answer 4.** The entropy goes down, and we did not lose any information (see the next cell). Why is the entropy lower? Why does it stop decreasing after a few levels?

*(your answer here)*

Now rebuild the image. One level is undone by adding $a$ back to the three remainders and by putting the pixels back at their place.

In [ ]:
def merge(R):
    """Undo one level of split()."""
    h, w = R.shape[0] // 2, R.shape[1] // 2
    a, b, c, d = R[:h, :w], R[:h, w:], R[h:, :w], R[h:, w:]
    # undo the remainders: b + a, c + a, d + a
    # Your code here
    im = np.zeros(R.shape, dtype=R.dtype)
    im[0::2, 0::2], im[0::2, 1::2], im[1::2, 0::2], im[1::2, 1::2] = a, b, c, d
    return im

def reconstruct(pyr, levels):
    """Rebuild the image from a pyramid of the given number of levels."""
    out = pyr.copy()
    sizes = [pyr.shape[0] // 2 ** k for k in range(levels)]     # size of the block split at each level
    # undo the levels from the last one to the first one
    # Your code here
    return out

for lev in [1, 3, 6]:
    rebuilt = reconstruct(pyramid(camera_u8, lev), lev)
    check('exact reconstruction with %d level(s)' % lev, np.array_equal(rebuilt, camera_u8))

## Optional challenge

Choose one:
- Apply a soft-tissue window (centre 40 HU, width 400 HU) and a bone window (centre 400 HU, width 1800 HU) to the CT slice, with a LUT. Display both.
- Reduce the camera image from 256 to 8 gray levels and describe the false contours you see.

In [ ]:
# CT windows with a LUT, or 8-level quantization
# Your code here

## To hand in
This notebook, completed, with the four answers written.